# Chapter 2 — NB 01: extração **independente** do ZNF175 no v1 (Freeze One)

**Princípio:** esta é a **nossa própria análise**, sem usar nenhum extrato, lista de variantes ou janela do **Daniel** ou do **Park**.
Definimos o locus a partir da **coordenada documentada do NCBI** e puxamos **todas** as variantes da janela (sem pré-filtro de MAF/pLOF).

**Locus (NCBI RefSeq, GRCh38):** `NC_000019.10:51,571,283-51,592,510` (+ strand) → no pVCF/PLINK o contig é `19`.

**Fonte de dados:** Freeze One **GL biallelic** PLINK (produto bruto do PMBB — *não* é do Daniel/Park):
`/static/PMBB/PMBB_Freeze17/genotype/exome/all_variants/UPENN_Freeze_One_GRCh38.GL.pVCF.biallelic`

**Ferramenta:** `plink2`. (bcftools/tabix não estão disponíveis neste ambiente.)

**Por que GL biallelic:** `GL` = QC de genótipo padrão do PMBB (QD/DP/AB) — **não** é filtro de pLOF/MAF, então respeita o "pull all, then filter"; e é comparável ao v2, que também é GL. (`NF` = totalmente sem filtro, disponível se quisermos o mais cru.)

> Saída: `results/v1_znf175_variants.csv` (nossa tabela de variantes do v1 com MAF e contagens de genótipo).

In [1]:
# --- Setup ---
import subprocess
from pathlib import Path
import pandas as pd

BASE    = Path("/project/hall/analysis/hearing-loss-genomics")
RESULTS = BASE / "analysis/chapter_2/results/01"
RESULTS.mkdir(parents=True, exist_ok=True)

PLINK2   = "/appl/plink2-20240804/plink2"
V1_BFILE = "/static/PMBB/PMBB_Freeze17/genotype/exome/all_variants/UPENN_Freeze_One_GRCh38.GL.pVCF.biallelic"

# NCBI RefSeq ZNF175 gene coordinates, GRCh38 (NC_000019.10), + strand
CHROM, START, END = "19", 51571283, 51592510
print(f"ZNF175 locus (NCBI GRCh38): chr{CHROM}:{START:,}-{END:,}  (span {END-START:,} bp)")
print("plink2:", PLINK2)

ZNF175 locus (NCBI GRCh38): chr19:51,571,283-51,592,510  (span 21,227 bp)
plink2: /appl/plink2-20240804/plink2


## 1. Extrair a janela e calcular frequências (plink2)

Recortamos a janela do NCBI no chr19 e pedimos ao plink2:
- `--freq` → frequência alélica (`.afreq`)
- `--geno-counts` → contagens de genótipo HOM_REF / HET / HOM_ALT / MISSING (`.gcount`)

Nenhum filtro de MAF/pLOF é aplicado aqui — pegamos **todas** as variantes da janela.

In [2]:
# --- Run plink2: extract NCBI window, compute allele freq + genotype counts ---
out_prefix = RESULTS / "v1_znf175_freeze_one"
cmd = [PLINK2,
       "--bfile", V1_BFILE,
       "--chr", CHROM, "--from-bp", str(START), "--to-bp", str(END),
       "--freq", "--geno-counts",
       "--memory", "8000", "--threads", "4",
       "--out", str(out_prefix)]
print("CMD:", " ".join(cmd), "\n")
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout[-1500:])
if res.returncode != 0:
    print("STDERR:\n", res.stderr[-1500:])
assert res.returncode == 0, "plink2 failed"
print("\nOutputs:", [p.name for p in RESULTS.glob('v1_znf175_freeze_one.*')])

CMD: /appl/plink2-20240804/plink2 --bfile /static/PMBB/PMBB_Freeze17/genotype/exome/all_variants/UPENN_Freeze_One_GRCh38.GL.pVCF.biallelic --chr 19 --from-bp 51571283 --to-bp 51592510 --freq --geno-counts --memory 8000 --threads 4 --out /project/hall/analysis/hearing-loss-genomics/analysis/chapter_2/results/01/v1_znf175_freeze_one 



9%70%70%71%72%72%73%74%74%75%76%76%77%78%78%79%80%80%81%82%82%83%84%84%85%86%86%87%88%88%89%90%90%91%92%92%93%94%94%95%96%96%97%98%98%99%
--freq: Allele frequencies (founders only) written to
/project/hall/analysis/hearing-loss-genomics/analysis/chapter_2/results/01/v1_znf175_freeze_one.afreq
.
--geno-counts: 0%0%1%1%2%3%3%4%5%5%6%7%7%8%9%9%10%11%11%12%13%13%14%15%15%16%17%17%18%19%19%20%21%21%22%23%23%24%25%25%26%27%27%28%29%29%30%31%31%32%33%33%34%35%35%36%37%37%38%39%39%40%41%41%42%43%43%44%45%45%46%47%47%48%49%49%50%50%51%52%52%53%54%54%55%56%56%57%58%58%59%60%60%61%62%62%63%64%

## 2. Montar a nossa tabela de variantes (MAF + portadores)

Juntamos `.afreq` e `.gcount` e derivamos:
- **MAF** = min(ALT_FREQS, 1 − ALT_FREQS)
- **carriers** = HET + HOM_ALT (pessoas com ≥1 alelo alternativo)
- **AC_alt** = HET + 2·HOM_ALT (contagem do alelo alternativo)
- **AN** = 2·(HOM_REF + HET + HOM_ALT) (alelos chamados)

In [3]:
# --- Build our v1 ZNF175 variant table ---
# plink2 columns: .gcount uses HET_REF_ALT_CTS (with S); both files carry a PROVISIONAL_REF? col.
af = pd.read_csv(str(out_prefix) + ".afreq", sep="\t")[
        ["#CHROM","ID","REF","ALT","ALT_FREQS","OBS_CT"]]
gc = pd.read_csv(str(out_prefix) + ".gcount", sep="\t")[
        ["#CHROM","ID","REF","ALT","HOM_REF_CT","HET_REF_ALT_CTS","TWO_ALT_GENO_CTS","MISSING_CT"]]

df = af.merge(gc, on=["#CHROM", "ID", "REF", "ALT"])
df["POS"]       = df["ID"].str.split(":").str[1].astype(int)
df["ALT_FREQS"] = pd.to_numeric(df["ALT_FREQS"], errors="coerce")
df["MAF"]       = df["ALT_FREQS"].apply(lambda x: min(x, 1 - x) if pd.notna(x) else x)
df["carriers"]  = df["HET_REF_ALT_CTS"] + df["TWO_ALT_GENO_CTS"]
df["AC_alt"]    = df["HET_REF_ALT_CTS"] + 2 * df["TWO_ALT_GENO_CTS"]
df["AN"]        = 2 * (df["HOM_REF_CT"] + df["HET_REF_ALT_CTS"] + df["TWO_ALT_GENO_CTS"])

cols = ["#CHROM","POS","ID","REF","ALT","ALT_FREQS","MAF","AC_alt","AN",
        "HOM_REF_CT","HET_REF_ALT_CTS","TWO_ALT_GENO_CTS","MISSING_CT","carriers","OBS_CT"]
df = df[cols].sort_values("POS").reset_index(drop=True)
df.to_csv(RESULTS / "v1_znf175_variants.csv", index=False)

print(f"Variants extracted in NCBI window (v1, GL biallelic): {len(df)}")
print(f"Position range observed: chr19:{df['POS'].min():,}-{df['POS'].max():,}")
print("saved -> results/v1_znf175_variants.csv")

Variants extracted in NCBI window (v1, GL biallelic): 151
Position range observed: chr19:51,573,255-51,588,539
saved -> results/v1_znf175_variants.csv


## 3. Visão geral (sem filtro vs. raras)

Quantas variantes, quantas seriam "raras" (MAF<0,1%), e as mais frequentes.
A classificação **pLOF** (que exige anotação independente, ex. VEP) e o **MAF por ancestralidade** ficam para os próximos passos.

In [4]:
# --- Overview ---
n          = len(df)
n_rare     = int((df["MAF"] < 0.001).sum())
n_carrier1 = int((df["carriers"] >= 1).sum())
print(f"total variants in window : {n}")
print(f"  MAF < 0.1% (rare)      : {n_rare}")
print(f"  with >=1 carrier       : {n_carrier1}")
print(f"  monomorphic (MAF==0)   : {int((df['MAF']==0).sum())}")

print("\nMost frequent variants in v1:")
print(df.sort_values("MAF", ascending=False)
        .head(10)[["POS","ID","REF","ALT","MAF","carriers","AC_alt","AN"]]
        .to_string(index=False))

summary = f'''# NB 01 — v1 (Freeze One) ZNF175 extraction (independent, NCBI coords)

- Locus: chr19:{START:,}-{END:,} (NCBI NC_000019.10, GRCh38, + strand)
- Source: Freeze One GL biallelic PLINK (PMBB raw); tool: plink2
- Variants in window (no MAF/pLOF filter): {n}
- Observed position range: chr19:{df['POS'].min():,}-{df['POS'].max():,}
- MAF<0.1% (rare): {n_rare}
- with >=1 carrier: {n_carrier1}

Output table: results/v1_znf175_variants.csv

Next:
- annotate independently (VEP/ANNOVAR) to flag pLOF / missense (REVEL)
- ancestry-stratified MAF (needs v1 ancestry labels)
- NB 02: harmonize & compare with v2 by chr:pos:ref:alt
'''
(RESULTS / "nb01_v1_extraction_summary.md").write_text(summary)
print("\n" + summary)

total variants in window : 151
  MAF < 0.1% (rare)      : 132
  with >=1 carrier       : 151
  monomorphic (MAF==0)   : 0

Most frequent variants in v1:
     POS              ID REF ALT      MAF  carriers  AC_alt    AN
51586847 19:51586847:T:C   T   C 0.238407      4783    5460 22902
51586553 19:51586553:T:C   T   C 0.216307      4408    4953 22898
51581842 19:51581842:G:A   G   A 0.095756      2071    2193 22902
51586761 19:51586761:C:T   C   T 0.094271      2036    2159 22902
51586729 19:51586729:C:G   C   G 0.077548      1515    1776 22902
51581583 19:51581583:C:T   C   T 0.071229      1406    1631 22898
51587418 19:51587418:T:A   T   A 0.010392       234     238 22902
51588529 19:51588529:A:C   A   C 0.010020       149     166 16566
51581735 19:51581735:C:T   C   T 0.004890       112     112 22902
51586968 19:51586968:G:A   G   A 0.003187        72      73 22902

# NB 01 — v1 (Freeze One) ZNF175 extraction (independent, NCBI coords)

- Locus: chr19:51,571,283-51,592,510 (NCBI NC_00